In [ ]:
import subprocess
import glob
from collections import Counter
import pathlib

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
BEDTOOLS = '/hpc/packages/minerva-common/BEDTools/2.27.1/bin/bedtools'

MERGED_DIR = pathlib.Path('./merged/')
MERGED_DIR.mkdir(exist_ok=True)

CONCAT_BED_PATH = MERGED_DIR / 'merged.filtered.bed'
INTERSECT_PATH = MERGED_DIR / 'merged.filtered.bed.intersect.gz'

# Read stat table to get cell types

In [ ]:
LEVEL = 'ageXclass'
BASE_PATH = '/sc/arion/projects/CommonMind/yeon/p/APA/run_SCAPTURE/'
STAT_PATH = BASE_PATH + f'1_make_pas/pl_{LEVEL}/peaks_5k-cells/stats/subsampled_read_counts.tsv'

CELL_TYPES = sorted(pd.read_table(STAT_PATH)['cell_type'])
CELL_TYPES

# Concat bed files from different samples

In [ ]:
PAS_PATH_FORMAT = BASE_PATH + f'3_merge_pas/pl_{LEVEL}/' + 'dup_gene_removed/{}.filtered_pas.rmdupgene.bed'

In [ ]:
tmp_file = 'bed.tmp'
subprocess.check_call(f'rm -rf {tmp_file}', shell=True)

for ct in CELL_TYPES:
    bed_path = PAS_PATH_FORMAT.format(ct)
    cmd = f"awk 'BEGIN {{OFS=\"\\t\"}} {{ $4 = $4 \"|{ct}\"; print }}' {bed_path}  >> {tmp_file}"
    subprocess.check_call(cmd, shell=True)

subprocess.check_call(f'cat {tmp_file} | sort -k1,1 -k2,2n - > {CONCAT_BED_PATH}', shell=True)
subprocess.check_call(f'rm -rf {tmp_file}', shell=True)


# Run bedtools intersect on concatenated bed file

In [ ]:
# Defines the cutoff of overlap between unique PASs
BED_OVERLAP_CUTOFF = 0.3

cmd = f"{BEDTOOLS} intersect -a {CONCAT_BED_PATH} -b {CONCAT_BED_PATH} -s -f {BED_OVERLAP_CUTOFF} -split -wao -sorted | gzip - > {INTERSECT_PATH}"
subprocess.check_call(cmd, shell=True)

# Load bedintersect

In [ ]:
columns = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 
           'blockCount', 'blockSizes', 'blockStarts', 'chromB', 'startB', 'endB', 'nameB', 'scoreB', 
           'strandB', 'thickStartB', 'thickEndB', 'itemRgbB', 'blockCountB', 'blockSizesB', 'blockStartsB', 'overlap']


In [ ]:
[(columns[col], 'str') for col in [0, 12, 18, 19, 20, 21]]

In [ ]:
len(columns)

In [ ]:
df_intersect = pd.read_table(INTERSECT_PATH, header=None, index_col=False, names=columns, compression='gzip',
                             dtype=dict([(col, 'category') for col in 'chrom strand chromB strandB'.split()]))

In [ ]:
# To reduce memory usage
cols_to_removed = 'score thickStart thickEnd itemRgb chromB scoreB strandB thickStartB thickEndB itemRgbB'.split()
for col in cols_to_removed:
    del df_intersect[col]

In [ ]:
df_intersect

In [ ]:
df_intersect.name.head().apply(print)

## Sanity check: check -f works or not. For some verions of bedtools, -f doesn't work if -split is set

In [ ]:
se_overlap_ratio = df_intersect['overlap'] / (df_intersect['blockSizes'].apply(lambda x: sum([int(bs) for bs in x[:-1].split(',')])))
se_overlap_ratio.hist()
plt.show()

In [ ]:
df_intersect['overlap'].min(), se_overlap_ratio.min()

In [ ]:
(df_intersect['nameB']=='.').sum()

## Just check overlap ratios in B

In [ ]:
se_overlap_ratio = df_intersect['overlap'] / (df_intersect['blockSizesB'].apply(lambda x: sum([int(bs) for bs in x[:-1].split(',')])))
se_overlap_ratio.hist()
plt.show()

se_overlap_ratio.min()

1. Assign group numbers to intersect
2. Read bed before intersect, assign group to them
3. Convert Bed12 to bed6, with group name
4. Split by groups -> Bedtools merge
6. Convert merged groups into single bed12 line. Maybe by gtf to bed?

# Assign group numbers to intersection

In [ ]:
name_a_to_b = dict(df_intersect.groupby('name')['nameB'].apply(set))
name_b_to_a = dict(df_intersect.groupby('nameB')['name'].apply(set))

In [ ]:
len(name_a_to_b), len(name_b_to_a)

In [ ]:
print('As Bedtools intersect graph is directed, so a to b and b to a are different')
print('Update name_a_to_b by name_b_to_a')

for k, v in name_a_to_b.items():
    if v != name_b_to_a[k]:
        print(k)
        print(v)
        print(name_b_to_a[k])
        break

for k, v in name_a_to_b.items():
    if v != name_b_to_a[k]:
        name_a_to_b[k] |= name_b_to_a[k]


In [ ]:
for k, v in name_a_to_b.items():
    if (v | name_b_to_a[k]) != v:
        print(k)
        print(v)
        print(name_b_to_a[k])
        break


## Recursion based group assignment

In [ ]:
group_id = 0
group_map = {}

def assign_group(name_a, name_a_to_b, group_map):
    for name_b in name_a_to_b[name_a]:
        if name_b not in group_map:
            group_map[name_b] = group_map[name_a]
            assign_group(name_b, name_a_to_b, group_map)
        # If all assigned, do nothing and escape the recursion
        

for name_a in name_a_to_b:
    overlap_nb_group_ids = [group_map[nb] for nb in name_a_to_b[name_a] if nb in group_map]
    
    # No need to check results; will do separate sanity check
    if name_a in group_map:
        assign_group(name_a, name_a_to_b, group_map)
    elif overlap_nb_group_ids: # If any of name_b already assigned to a group
        group_map[name_a] = overlap_nb_group_ids.pop()
        assign_group(name_a, name_a_to_b, group_map)
    else:
        group_id += 1
        group_map[name_a] = group_id
        assign_group(name_a, name_a_to_b, group_map)
        


In [ ]:
len(group_map)

In [ ]:
df_intersect['group'] = df_intersect['name'].map(group_map)
df_intersect

# Sanity check: group of name B should be equal to the group of name A

In [ ]:
df_intersect[df_intersect.nameB.map(group_map)!=df_intersect.group]

# Assign group numbers to bed file

In [ ]:
columns = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 
           'blockCount', 'blockSizes', 'blockStarts']

GROUP_NAMED_BED_PATH = 'named_by_groups.bed.tmp'

In [ ]:
df_bed = pd.read_table(CONCAT_BED_PATH, header=None, index_col=False, names=columns,
                       dtype=dict([(col, 'category') for col in 'chrom strand'.split()]))
df_bed

In [ ]:
df_bed['name'].duplicated().sum()

In [ ]:
# Copy original bed
df_bed_org = df_bed.copy()

df_bed['name'] = df_bed['name'].map(group_map)
df_bed.to_csv(GROUP_NAMED_BED_PATH, header=False, index=False, sep='\t')

# Convert bed12 to bed6 (split exons into separate rows)

In [ ]:
BED6_PATH = 'bed6.bed.tmp'

cmd = f'{BEDTOOLS} bed12tobed6 -i {GROUP_NAMED_BED_PATH} |\
sort -k1,1 -k2,2n > {BED6_PATH}'

subprocess.check_call(cmd, shell=True)


In [ ]:
df_b6 = pd.read_table(BED6_PATH, header=None, index_col=False, dtype={0: 'category'})
df_b6.columns = 'chrom start end group score strand'.split()
df_b6.head()

# Split each groups and run bedtools merge

In [ ]:
df_b6.tail(20)

In [ ]:
SPLIT_BED_TMP = 'split_group_bed.tmp'
df_b6[df_b6.group==8999].to_csv(SPLIT_BED_TMP, header=False, index=False, sep='\t')

cmd = f'{BEDTOOLS} merge -s -c 6,4 -o distinct,distinct -i {SPLIT_BED_TMP}'
output = subprocess.check_output(cmd, shell=True)
#ret = [line.decode().strip().split('\t') for line in .stdout.readlines()]
output.decode().strip().split('\n')

In [ ]:
# Unique groups; No need to run bedtools
print(df_b6[~df_b6.group.duplicated(keep=False)].group.nunique(), df_b6.group.nunique())
df_b6[~df_b6.group.duplicated(keep=False)]

In [ ]:
bed_merged_lines = []
col_orders = 'chrom start end strand group'.split()

for group, df_eg in df_b6.groupby('group'):
    if len(df_eg)==1:
        se_row = df_eg.iloc[0]
        line = '\t'.join([str(se_row[col]) for col in col_orders])
        bed_merged_lines.append(line)

    else:
        df_eg.to_csv(SPLIT_BED_TMP, header=False, index=False, sep='\t')
        
        cmd = f'{BEDTOOLS} merge -s -c 6,4 -o distinct,distinct -i {SPLIT_BED_TMP}'
        output = subprocess.check_output(cmd, shell=True)
        #ret = [line.decode().strip().split('\t') for line in .stdout.readlines()]
        bed_merged_lines.extend(output.decode().strip().split('\n'))



In [ ]:
df_merged = pd.DataFrame([line.split('\t') for line in bed_merged_lines])
df_merged.columns = 'chrom start end strand group'.split()
df_merged.head()

In [ ]:
df_merged['start'] = df_merged.start.astype(int)
df_merged['end'] = df_merged.end.astype(int)

In [ ]:
# All merged chunk has unique group
df_merged[df_merged.group.apply(lambda x: ',' in x)]

In [ ]:

def merge_bed_blocks_to_one_line(df_in):
    new_chrom = df_in.iloc[0].chrom
    new_start = df_in.start.min()
    new_end = df_in.end.max()
    
    block_sizes = ",".join(map(str, (df_in.end - df_in.start).tolist())) + ','
    block_starts = ",".join(map(str, (df_in.start - new_start).tolist())) + ','
    block_count = len(df_in)

    row = [
        new_chrom,
        new_start,
        new_end,
        df_in.group.iloc[0],
        0,
        df_in.strand.iloc[0],
        new_start,
        new_end,
        0,
        block_count,
        block_sizes,
        block_starts,
    ]
    return row

merged_bed12_lines = df_merged.groupby('group')[df_merged.columns].apply(merge_bed_blocks_to_one_line)
merged_bed12_lines
        

In [ ]:
df_merged_bed12 = pd.DataFrame(merged_bed12_lines.tolist())
df_merged_bed12

In [ ]:
(df_merged_bed12[2] - df_merged_bed12[1]).describe()

In [ ]:
df_merged_bed12[(df_merged_bed12[2] - df_merged_bed12[1]) == 584220]

In [ ]:
df_merged_bed12[10].apply(lambda x: sum([int(v) for v in x[:-1].split(',')])).describe()

In [ ]:
df_merged_bed12['block_size_list'] = df_merged_bed12[10].apply(lambda x: [int(v) for v in x[:-1].split(',')])

fig, ax = plt.subplots()

bins = list(range(1, 10))
df_merged_bed12.block_size_list.apply(len).hist(ax=ax, bins=bins + [np.inf], zorder=10)
ax.set_xticks(bins)
ax.set_xlabel('Number of exons')
ax.set_ylabel('Number of PAS groups')
ax.set_xticks(np.array(bins) + 0.5)
ax.set_xticklabels(bins)

ax.set_title(f'# Exon per merged PAS\n{LEVEL}-level')
plt.show()

In [ ]:
fig, ax = plt.subplots()

bins = list(range(1, 10))

se_blk_sum = df_merged_bed12.block_size_list.apply(sum).sort_values()

ax.plot(se_blk_sum, se_blk_sum.rank()/len(se_blk_sum), zorder=10)

ax.set_xlabel('PAS size (exon only, bp)')
ax.set_ylabel('Cumulative fraction')

ax.grid()

ax.set_title(f'Size of merged PASs\n{LEVEL}-level')
plt.show()

In [ ]:
print(se_blk_sum.quantile(0.95), se_blk_sum.quantile(0.99), se_blk_sum.quantile(0.999))
se_blk_sum.describe()

In [ ]:
se_blk_sum.to_pickle(f'../pas_sizes_{LEVEL}.pkl')

# Assign gene name to group, choose the best matched gene name as the same priority used in gene-level duplicate removal

In [ ]:
df_gene_dup_removal_info = pd.read_pickle(f'./PAS_duplicated_from_many_genes_removal_info-{LEVEL}.pkl')
df_gene_dup_removal_info['name'] = df_gene_dup_removal_info['name'] + '|' + df_gene_dup_removal_info['cellType']
df_gene_dup_removal_info.head()

In [ ]:
removed_cols = 'chrom start end score strand thickStart thickEnd itemRgb blockCount blockStarts blockSizes'.split()
for col in removed_cols:
    del df_gene_dup_removal_info[col]
df_gene_dup_removal_info.head()

In [ ]:
# Some PAS groups has multiple gene names

df_name_and_group = pd.DataFrame([(k, v) for k, v in group_map.items()], columns='name group'.split())
df_name_and_group['gene_name'] = df_name_and_group['name'].str.split('|').apply(lambda x: x[0])
df_name_and_group

In [ ]:
df_name_group_info = pd.merge(df_name_and_group, df_gene_dup_removal_info, left_on='name', right_on='name', how='left')
# All name has info
print(df_name_group_info['psychAdCpm'].isnull().sum())
df_name_group_info

In [ ]:
df_name_group_info.sort_values(by='group psychAdCpm longReadCpm regionPriority geneLength'.split(), 
                                   ascending=[True, False, False, True, False], inplace=True)

df_name_group_info.to_csv('PAS_merged_from_many_samples_info.tsv.gz', compression='gzip', sep='\t')
df_name_group_info.to_pickle('PAS_merged_from_many_samples_info.pkl')

In [ ]:
df_pas_group_name_uniq = df_name_group_info.drop_duplicates('group')
group_to_primary_gene_name = dict(zip(df_pas_group_name_uniq['group'], df_pas_group_name_uniq['gene_name']))
len(group_to_primary_gene_name)

In [ ]:
df_merged_bed12['gene_name'] = df_merged_bed12[3].astype(int).map(group_to_primary_gene_name)
df_merged_bed12['name'] = df_merged_bed12['gene_name'] + '|' + df_merged_bed12[3]
df_merged_bed12

In [ ]:
BED_OUTPUT_PATH = f'merged/all_passed_merged_PAS-{LEVEL}.bed'

df_merged_bed12[[0, 1, 2, 'name', 4, 5, 6, 7, 8, 9, 10, 11]].sort_values(by=[0, 1, 2]).to_csv(BED_OUTPUT_PATH, sep='\t', header=False, index=False)

# How many PASs per each gene?

In [ ]:
df_merged_bed12

In [ ]:
len(df_merged_bed12) / len(set(df_merged_bed12['gene_name']))

In [ ]:


fig, ax = plt.subplots(figsize=(4.5, 3))

bins = list(range(1, 17))
df_merged_bed12.groupby('gene_name')[0].count().hist(ax=ax, bins=bins+[np.inf], zorder=10)
ax.set_xlabel('Number of PASs')
ax.set_ylabel('Number of genes')
ax.set_xticks(np.array(bins[:-1]) + 0.5)
ax.set_xticklabels(bins[:-1])

ax.set_title(f'# PAS per gene\n{LEVEL}-level')
plt.show()

In [ ]:
se_cnt = df_merged_bed12.groupby('gene_name')[0].count()

In [ ]:
(se_cnt>1).sum(), len(se_cnt), (se_cnt>1).sum() / len(se_cnt)

# Sanity check: visualize merged block

In [ ]:
df_bed_org['group'] = df_bed['name']

# Save this one for further analysis
df_bed_org.to_pickle(f'Each_PAS_with_group_{LEVEL}.pkl')

In [ ]:
df_merged_bed12['total_size'] = df_merged_bed12.block_size_list.apply(sum)
df_merged_bed12.sort_values('total_size', ascending=False).head(10)

In [ ]:

def draw_group(group, pdf=None, png=None):
    df_vis = df_bed_org[df_bed_org['group']==group].sort_values('start')
    gene_name = group_to_primary_gene_name[group]

    fig, ax = plt.subplots(figsize=(12, 0.25 * len(df_vis)))
    cmap = plt.get_cmap('tab20')
    norm = mcolors.Normalize(vmin=0, vmax=len(df_vis))
               
    pas_names = ['Merged']
    se_merged_vis = df_merged_bed12[df_merged_bed12[3]==str(group)].iloc[0]
    total_size = se_merged_vis.total_size
    bl_starts = se_merged_vis[1] + np.array([int(x) for x in se_merged_vis[11][:-1].split(',')])
    bl_ends = bl_starts + np.array([int(x) for x in se_merged_vis[10][:-1].split(',')])
    for bl_start, bl_end in zip(bl_starts, bl_ends):
        ax.plot([bl_start, bl_end], [0, 0], lw=2, color='magenta')
    
    for n, (idx, se_row) in enumerate(df_vis.iterrows()):
        pas_name = se_row['name']
        split_names = pas_name.split('|')
        pas_names.append(split_names[0] + ' ' + split_names[-1])
        
        bl_starts = se_row.start + np.array([int(x) for x in se_row.blockStarts[:-1].split(',')])
        bl_ends = bl_starts + np.array([int(x) for x in se_row.blockSizes[:-1].split(',')])
    
        for bl_start, bl_end in zip(bl_starts, bl_ends):
            ax.plot([bl_start, bl_end], [n+1, n+1], lw=2, color=cmap(norm(n)))
      
    #ax.legend(loc=(1.01, 0.0))
    
    ax.set_title(f'Overlap of {gene_name} PAS (group {group}), Merged={total_size} bp')
    yticks = list(range(len(pas_names)))
    ax.set_yticks(list(range(len(pas_names))))
    ax.set_yticklabels(pas_names)
    ax.set_ylim((yticks[-1] + 1, yticks[0] - 1))
    ax.set_xlabel(f'Position in chromosome {se_row.chrom}')
    if pdf:
        plt.savefig(pdf, format='pdf', bbox_inches='tight')
    if png:
        plt.savefig(png, format='png', bbox_inches='tight')
    plt.show()

draw_group(1000)

In [ ]:
pathlib.Path('png').mkdir(exist_ok=True)

for gr in df_merged_bed12.sort_values('total_size', ascending=False).head(20)[3]:
    draw_group(int(gr), png=f'png/group_{gr}_merged_no_cpm_cutouff.png')